# Install packages and write functions

In [1]:
import os
import pandas as pd
def count_files(directory):
    total_files = 0
    for root, dirs, files in os.walk(directory):
        total_files += len(files)  # Add count of files in this directory
    return total_files

# Replace 'your_directory_path' with the path to the directory you want to check
# directory_path = "/Volumes/she4/knowledgegraph/biobert_aioner/AIONER/example/input/"
# count_files(directory_path)

import transformers
from transformers import AutoModelForTokenClassification, AutoTokenizer
import torch

/Users/she4/miniconda3/envs/myenv/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def extract_abstract(pubtator_file):
    abstract_text = ""
    with open(pubtator_file, 'r', encoding='utf-8') as file:  # Ensure proper encoding is used
        for line in file:
            parts = line.strip().split('|')
            if len(parts) == 3 and parts[1].strip() == 'a':  # Checks if it's the abstract line
                abstract_text = parts[2].strip()
                break  # Assuming only one abstract per file
    return abstract_text

# Function to align labels with input tokens
def align_predictions(predictions, label_list,tokenizer):
    entities = []
    for idx, pred in enumerate(predictions[0]):
        label = label_list[pred.item()]
        entities.append((tokenizer.decode(tokenized_input["input_ids"][0][idx]), label))
    return entities

def process_entities(named_entities):
    processed_entities = []
    current_entity = ""
    
    for token, tag in named_entities:
        is_subword = token.startswith('##')
        token_clean = token.replace('##', '')
        
        if tag.startswith('B') and not is_subword:
            if current_entity:
                processed_entities.append(current_entity.strip())
            current_entity = token_clean
        else:
            # If token is a continuation (either proper 'I-*' or 'B-*' but with ##), append without space
            if is_subword or tag.startswith('I'):
                current_entity += token_clean
            elif token in ['-', '(', ')']:
                current_entity += token
            else:
                current_entity += ' ' + token_clean

    if current_entity:
        processed_entities.append(current_entity.strip())

    return processed_entities


def extract_entities(pubtator_file):
    entities = []
    with open(pubtator_file, 'r', encoding='utf-8') as file:
        for line in file:
            parts = line.strip().split('\t')
            if len(parts) >= 5 and parts[0].isdigit():  # Check if it looks like an entity line
                entity_info = {
                    'PMID': parts[0],
                    'OffsetStart': parts[1],
                    'OffsetEnd': parts[2],
                    'Entity': parts[3],
                    'Text': parts[4],
                    'Metadata': parts[5] if len(parts) > 5 else None
                }
                entities.append(entity_info)
    return entities

import re

def has_exact_match(term, original_text):
    term_norm = term.lower().replace(" ", "")
    text_norm = original_text.lower().replace(" ", "")
    
    return term_norm in text_norm

In [ ]:
import csv
import re
import fitz
from groq import Groq
import os
import itertools

client = Groq(api_key='')
def generate_prompts_disease(potential_disease):
    system_prompt = """
    You are a doctor with expertise in disease naming conventions. 
    """
    user_prompt = f"""
    I have these terms, {potential_disease}, 
    can you shorten the list to retain the most informative disease terms only. 
    Please only output the term?
    """
    return system_prompt, user_prompt

import time
def get_response_disease(potential_pathways):
    retry_count = 0
    max_retries = 3  # adjust the number of retries as needed
    delay = 300  # 5 minutes in seconds

    while retry_count < max_retries:
        try:
            system_prompt, user_prompt = generate_prompts_disease(potential_pathways)
            response = client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    }
                ],
                model="llama3-70b-8192", #"llama3-groq-70b-8192-tool-use",
                max_tokens=8192,
                temperature=0
            )
            return response.choices[0].message.content
        except Exception as e:
            if "503" in str(e) or "internal_server_error" in str(e):
                print(f"Retry {retry_count+1}: {e}")
                time.sleep(delay)
                retry_count += 1
            else:
                raise

    raise Exception("Maximum retries exceeded")

In [ ]:
import spacy
import re
from spacy.matcher import Matcher
nlp = spacy.load("en_core_web_sm")
def extract_pathways(sentences):
    nlp = spacy.load("en_core_web_sm")
    pathways = set()

    for sentence in sentences:
        doc = nlp(sentence)
        
        # Extract noun phrases
        for chunk in doc.noun_chunks:
            # Check if the chunk has more than one token and doesn't contain stop words
            if len(chunk) > 1 and not any(token.is_stop for token in chunk):
                # Add the noun phrase to the pathways set
                pathways.add(chunk.text)

    return list(pathways)
def split_into_sentences(paragraph):
    sentences = re.split(r'(?<!\w\.\w.)(?<![A-Z][a-z]\.)(?<=\.|\?)\s', paragraph)
    return sentences

import requests
from urllib.parse import quote_plus
import Levenshtein
import re

# You'll need to replace this with your actual UMLS API key
API_KEY = ""
BASE_URL = "https://uts-ws.nlm.nih.gov/rest"

def query_umls(term):
    encoded_term = quote_plus(term)
    url = f"{BASE_URL}/search/current?string={encoded_term}&apiKey={API_KEY}"
    
    response = requests.get(url)
    if response.status_code == 200:
        return response.json()['result']['results']
    else:
        print(f"Error querying UMLS for term '{term}': {response.status_code}")
        return []

def is_similar_term(term1, term2, threshold=0.8):
    # Normalize terms: lowercase and remove extra spaces
    term1 = ' '.join(term1.lower().split())
    term2 = ' '.join(term2.lower().split())
    
    # Calculate Levenshtein ratio
    ratio = Levenshtein.ratio(term1, term2)
    
    return ratio >= threshold

def is_biomedical_term(term,threshold=0.90):
    results = query_umls(term)
    
    if results:
        # Check if any result is similar to our term
        return any(is_similar_term(term, result['name'], threshold) for result in results)
    
    # If no exact match for the full term, check individual words
    words = term.split()
    return any(query_umls(word) for word in words if len(word) > 2)

def filter_biomedical_terms(entities):
    biomedical_terms = []
    removed_terms = []
    
    for term in entities:
        if is_biomedical_term(term):
            biomedical_terms.append(term)
        else:
            removed_terms.append(term)
    
    return biomedical_terms, removed_terms

def is_likely_biology_term(term):
    # Convert to lowercase for comparison
    term_lower = term.lower()
    words = term_lower.split()
    
    # Check for terms with specific characteristics common in biology
    
    # Terms with numbers or Greek letters
    if re.search(r'\d', term) or re.search(r'alpha|beta|gamma|delta', term_lower):
        return True
    
    # Terms with hyphens or slashes
    if '-' in term or '/' in term:
        return True
    
    # Terms with uppercase letters (potential acronyms)
    if re.search(r'\b[A-Z]{2,}\b', term):
        return True
    
    # Longer terms (4 or more words) are more likely to be specific
    if len(words) >= 4:
        return True
    
    # Terms ending with specific suffixes common in biology
    biology_suffixes = ['ase', 'in', 'ion', 'itis', 'osis', 'oma', 'icity', 'pathy']
    if any(word.endswith(tuple(biology_suffixes)) for word in words):
        return True
    
    # Terms containing specific substrings common in biology
    biology_substrings = ['cell', 'gene', 'protein', 'receptor', 'antibody', 'immune', 'tumor', 'cancer', 'virus', 'bacteria']
    if any(substring in term_lower for substring in biology_substrings):
        return True
    
    return False

def clean_terms(terms, original_text):
    original_text_lower = original_text.lower()
    
    def has_exact_match(term):
        pattern = r'\b' + re.escape(term.lower()) + r'\b'
        return bool(re.search(pattern, original_text_lower))
    
    # Filter terms that have an exact match and are likely biology terms
    filtered_terms = [term for term in terms if has_exact_match(term) and is_likely_biology_term(term)]
    
    sorted_terms = sorted(filtered_terms, key=len, reverse=True)
    cleaned_terms = []

    for i, term in enumerate(sorted_terms):
        is_substring = False
        for j, longer_term in enumerate(sorted_terms):
            if i != j and term.lower() in longer_term.lower():
                is_substring = True
                break
        if not is_substring:
            cleaned_terms.append(term)

    return cleaned_terms

def remove_genes(processed_gene, cleaned_terms):
    # Convert processed_gene to lowercase
    processed_gene_lower = set(word.lower() for word in processed_gene)
    
    # Function to check if a term contains any word from processed_gene
    def should_keep_term(term):
        term_lower = term.lower()
        for gene in processed_gene_lower:
            if gene in term_lower:
                # print(f"Removing '{term}' because it contains '{gene}'")
                return False
        return True
    
    # Keep only the terms that don't contain any word from processed_gene
    result = [term for term in cleaned_terms if should_keep_term(term)]
    
    return result

import string
def filter_entities(entities):
    def should_keep(entity):
        digit_count = sum(c.isdigit() for c in entity)
        has_punctuation = any(c in string.punctuation for c in entity)
        return digit_count < 3 and not has_punctuation
    
    return [entity for entity in entities if should_keep(entity)]

def filter_entities(entities):
    return [entity for entity in entities if '(' not in entity and ')' not in entity]

import csv
import re
import fitz
from groq import Groq
import os
import itertools

# Initialize the Groq API client
client = Groq(api_key='')

def generate_prompts_pathway(potential_pathways):
    system_prompt = """
    You are a computational biologist with expertise in biomedical pathways. 
    Your task is to identify genuine biomedical pathways from a given list of terms. 
    Use your knowledge to filter out non-pathway terms based on the following criteria:

    1. Exclude laboratory techniques, methods, or procedures (e.g., assays).
    2. Exclude terms that are too general (e.g., "gene expression", "transcription activity", "transcriptional activation",'regulation').
    3. Exclude cell types, cell lines, or cell clones.

    Provide only the terms that represent valid biomedical pathways. Double-check your output to ensure all terms meet the criteria.
    """

    user_prompt = f"""
    I've provided these terms: {potential_pathways}.
    Please identify which of these are likely biomedical pathways, excluding any that don't meet the criteria specified in your instructions.
    Output only the valid pathway terms (in a list) with no additional explanation.
    """

    return system_prompt, user_prompt

def get_response_pathway(potential_pathways):
    system_prompt, user_prompt = generate_prompts_pathway(potential_pathways)
    
    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
        model="llama3-70b-8192", #"llama3-groq-70b-8192-tool-use",
        max_tokens=8192,
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
import csv
import re
import fitz
from groq import Groq
import os
import itertools

# Initialize the Groq API client
client = Groq(api_key='')
def generate_prompt_finalize(entities):
    system_prompt = f"""
    I have a list of terms that an algorithm has extracted as potential biologically relevant entities from an abstract.
    As a biologist, can you please help me remove uninformative or redundant terms, and return only the informative ones?
    IMPORTANT: For each remaining term, please preserve the original format and include its entity type as one of: gene, disease, celltype, or pathway.
    Format the output as a Python list of (term, entity_type) tuples. Do not rephrase or polish the terms. Do not include any explanation.
    """

    user_prompt = f"""
    Potential biomedically relevant terms: {', '.join(entities)}
    Please output only the informative terms in a list! No explanation!
    """
    return system_prompt, user_prompt

def get_response_finalize(entities):
    system_prompt, user_prompt = generate_prompt_finalize(entities)
    response = client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
        model="llama3-70b-8192", #"llama3-groq-70b-8192-tool-use",
        max_tokens=8192,
        temperature=0
    )
    return response.choices[0].message.content

In [ ]:
import itertools
from groq import Groq
import time

relationship_client = Groq(api_key='')

def generate_prompt_relationship(abstract, entity_pairs):
    system_prompt = """
    You are a computational biologist who excels at carefully reading biomedical publication abstracts and inferring the relationship between
    biomedical terms based on the abstract's content.
    """
    user_prompt = f"""
    I will provide you with an Abstract and multiple pairs of Biomedical terms.
    Abstract: {abstract}

    For each pair of terms, determine if there is a directional relationship from the first term to the second term.
    The relationship should be one of the following three options:
    - 'Inhibit' if the first term directly or indirectly reduces, blocks, or inhibits the second term.
    - 'Activate' if the first term directly or indirectly promotes or leads to the second term.
    - 'no association' if there is no clear direct or indirect relationship from the first term to the second term, or if you are unsure about the relationship based on the provided information.

    Important rules to follow:
    - The relationship must be inferred in the direction from the first term to the second term only. Do not infer relationships in the opposite direction unless explicitly stated in the sentence.
    - If the relationship is ambiguous or unclear in the abstract, choose 'no association.'
    - If either term lacks a clear biological meaning in the context of the abstract, output 'no association.'
    - After determining the relationship, validate the answer by re-reading the abstract. If you cannot confidently validate the direction of the relationship, then change your output to 'no association'.

    Biomedical term pairs:
    {', '.join([f'({pair[0]}, {pair[1]})' for pair in entity_pairs])}

    **Remember:**
    - If term1 inhibits term2, the output should be (term1, term2, Inhibit).
    - If term1 activates term2, the output should be (term1, term2, Activate).
    - If term1 does not affect term2 or you cannot validate it, output (term1, term2, no association).
    - If term2 does not directly or indirectly affect term1, the output should always be (term2, term1, no association).
    Output your response in the following format: ('term1', 'term2', relationship) and this format only.
    Examples:
    - "TNF-α stimulates NF-κB activity, resulting in increased expression of proinflammatory genes."
       - If term1 is TNF-α and term2 is NF-κB, the output is (TNF-α, NF-κB, Activate).
       - If term1 is NF-κB and term2 is TNF-α, the output is (NF-κB, TNF-α, no association).
    - "PD-1 signaling inhibits T cell activation, leading to immune evasion by tumor cells."
       - If term1 is PD-1 and term2 is T cell activation, the output is (PD-1, T cell activation, Inhibit).
       - If term1 is T cell activation and term2 is PD-1, the output is (T cell activation, PD-1, no association).

    Output your response in the following format for each pair:
    (term1, term2, relationship)

    Provide your answer as a comma-separated list of these tuples, one for each input pair.
    Only include pairs where the relationship is either 'Activate' or 'Inhibit'.
    """
    return system_prompt, user_prompt

def get_relationships(abstract, entities):
    pairs = list(itertools.permutations(entities, 2))
    
    system_prompt, user_prompt = generate_prompt_relationship(abstract, pairs)
    
    max_retries = 3
    retry_count = 0
    
    while retry_count < max_retries:
        try:
            response = relationship_client.chat.completions.create(
                messages=[
                    {
                        "role": "system",
                        "content": system_prompt,
                    },
                    {
                        "role": "user",
                        "content": user_prompt,
                    }
                ],
                model="llama3-70b-8192", # llama-3.1-70b-versatile,llama3-70b-8192
                max_tokens=8000,
                temperature=0
            )
            break  # If successful, break the retry loop
            
        except Exception as e:
            if "Error code: 400" in str(e):
                retry_count += 1
                if retry_count < max_retries:
                    print(f"Got error 400, waiting 2 minutes before retry {retry_count}/{max_retries}...")
                    time.sleep(300)  # Wait 5 minutes
                else:
                    print(f"Failed after {max_retries} retries")
                    raise  # Re-raise the error if we've exhausted retries
            else:
                raise  # Re-raise if it's not a 400 error
    
    # Parse the response
    relationships_text = response.choices[0].message.content
    relationships_list = []
    
    # Split the response into individual relationships
    for rel in relationships_text.split('),'):
        rel = rel.strip().strip('(').strip(')')
        if rel:
            parts = rel.split(',')
            if len(parts) == 3:
                term1, term2, relationship = [p.strip().strip("'").strip('"') for p in parts]
                if relationship in ['Activate', 'Inhibit']:
                    relationships_list.append((term1, term2, relationship))
    
    return relationships_list

In [7]:
import ast
import re

def process_large_list_string(input_string):
    input_string = input_string.strip()[1:-1]
    items = re.findall(r'\'[^\']*\'|[^,]+', input_string)
    items = [item.strip().strip("'") for item in items]
    unique_items = []
    seen = set()
    for item in items:
        if item not in seen:
            seen.add(item)
            unique_items.append(item)
    return unique_items

def process_input(input_string):
    try:
        # First, try to evaluate the string as a Python literal
        result = eval(input_string) # First, try to evaluate the string as a Python literal
        if not isinstance(result, list):
            raise ValueError("Input is not a list")
        return list(dict.fromkeys(result))  # Remove duplicates while preserving order
    except SyntaxError as e:
        if "EOL while scanning string literal" in str(e):
            print("Encountered 'EOL while scanning string literal' error. Switching to string processing method.")
            return process_large_list_string(input_string)
        else:
            raise  # Re-raise the exception if it's a different SyntaxError
    except Exception as e:
        print(f"An error occurred: {e}")
        print("Switching to string processing method.")
        return process_large_list_string(input_string)

# Construct the graph

### Load the three NER models

In [8]:
model_path_gene = "/Volumes/she4/knowledgegraph/result/output/BioNLP13PC"
model_gene = AutoModelForTokenClassification.from_pretrained(model_path_gene)
tokenizer_gene = AutoTokenizer.from_pretrained(model_path_gene)

model_path_disease = "/Volumes/she4/knowledgegraph/result/output/NCBI-disease"
model_disease = AutoModelForTokenClassification.from_pretrained(model_path_disease)
tokenizer_disease = AutoTokenizer.from_pretrained(model_path_disease)

model_path_celltype = "/Volumes/she4/knowledgegraph/result/output/GENIAcelltype"
model_celltype = AutoModelForTokenClassification.from_pretrained(model_path_celltype)
tokenizer_celltype = AutoTokenizer.from_pretrained(model_path_celltype)

### Run NER tasks and output relationship csv file for each abstract

##### Notes: 50 abstracts without debugging cost about 5 cents, then 24K abstracts (5 years worth of published abstracts) cost about $24

In [ ]:
directory = "/Volumes/she4/knowledgegraph/biobert_aioner/AIONER/example/input_more/"
files = [f for f in os.listdir(directory) if not f.startswith('.')]

processed = 0
for number in range(0,len(files)): 

    pmid = files[number]

    if os.path.isfile(os.path.join('/Volumes/she4/knowledgegraph/result/new_extracted_entities_from_NK_abstracts/', pmid))== True:
        continue # if already exist, then next abstract
    else:
        processed += 1
        file = os.path.join("/Volumes/she4/knowledgegraph/biobert_aioner/AIONER/example/input_more/",pmid)
        text = extract_abstract(file)

        try:
            tokenized_input = tokenizer_gene(text, padding=True, truncation=True, return_tensors="pt")
            model_gene.eval()
            with torch.no_grad():
                output = model_gene(**tokenized_input)
            text = text
        except RuntimeError as e:
            if "The size of tensor a" in str(e) and "must match the size of tensor b" in str(e):
                text = ' '.join(text.split()[:100])

        tokenized_input = tokenizer_gene(text, padding=True, truncation=True, return_tensors="pt")
        model_gene.eval()
        with torch.no_grad():
            output = model_gene(**tokenized_input)
        predictions = torch.argmax(output.logits, dim=2)
        label_list = ["B-Gene_or_gene_product","I-Gene_or_gene_product","I-Complex", "I-Cellular_component", "B-Cellular_component", "B-Complex", "B-Simple_chemical", "O","I-Simple_chemical"] # BC2GM, NCBI-disease
        # label_list = ["B","I","O"] # BC2GM, NCBI-disease
        aligned_predictions = align_predictions(predictions, label_list,tokenizer=tokenizer_gene)
        named_entities = [entity for entity in aligned_predictions if entity[1] != "O"]
        processed_entities_gene = process_entities(named_entities)
        # processed_entities_gene = [word for sublist in processed_entities_gene for word in sublist.split()]
        def connect_words(word_list):
            result = []
            i = 0
            while i < len(word_list):
                if word_list[i].endswith('-') and i + 1 < len(word_list):
                    result.append(word_list[i] + word_list[i+1])
                    i += 2
                else:
                    result.append(word_list[i])
                    i += 1
            return result
        processed_gene = connect_words(processed_entities_gene)
        processed_gene = [element.replace(' ', '') for element in processed_gene]
        processed_gene

        tokenized_input = tokenizer_disease(text, padding=True, truncation=True, return_tensors="pt")
        model_disease.eval()
        with torch.no_grad():
            output = model_disease(**tokenized_input)
        predictions = torch.argmax(output.logits, dim=2)
        label_list = ["B","I","O"] # BC2GM, NCBI-disease
        aligned_predictions = align_predictions(predictions, label_list,tokenizer=tokenizer_disease)
        named_entities = [entity for entity in aligned_predictions if entity[1] != "O"]
        processed_entities_disease = process_entities(named_entities)
        processed_entities_disease = [word for sublist in processed_entities_disease for word in sublist.split()]
        processed_disease = []
        for i in processed_entities_disease:
            if has_exact_match(i,text):
                processed_disease.append(i)
        # processed_disease = set(processed_disease)
        if len(processed_disease)!=0:
            processed_disease = get_response_disease(processed_disease)
            processed_disease = set(processed_disease.split(", "))
            processed_disease = list(processed_disease)
        processed_disease = processed_disease
        processed_disease

        tokenized_input = tokenizer_celltype(text, padding=True, truncation=True, return_tensors="pt")
        model_celltype.eval()
        with torch.no_grad():
            output = model_celltype(**tokenized_input)
        predictions = torch.argmax(output.logits, dim=2)
        label_list = ["I","O","B"] 
        aligned_predictions = align_predictions(predictions, label_list,tokenizer=tokenizer_celltype)
        named_entities = [entity for entity in aligned_predictions if entity[1] != "O"]
        processed_entities_celltype = process_entities(named_entities)
        # processed_entities_celltype = [word for sublist in processed_entities_celltype for word in sublist.split()]
        processed_celltype = []
        for i in processed_entities_celltype:
            if has_exact_match(i,text):
                processed_celltype.append(i)
        processed_celltype = set(processed_celltype)
        processed_celltype = list(processed_celltype)

        sentences_list = split_into_sentences(text)
        pathway_terms = extract_pathways(sentences_list)
        biomedical_terms, removed_terms = filter_biomedical_terms(entities = pathway_terms)
        cleaned_terms = clean_terms(biomedical_terms, text)
        cleaned_terms = filter_entities(cleaned_terms)
        pathways = remove_genes(processed_gene, cleaned_terms)
        pathways = get_response_pathway(pathways)
        # pathways = eval(pathways)
        if 'valid' in pathways: 
            pathways = []
        else:
            pathways = process_input(pathways)

        # nested_list = [[word for sublist in processed_disease for word in sublist.split()],
        #             [word for sublist in processed_gene for word in sublist.split()],
        #             [word for sublist in processed_celltype for word in sublist.split()],
        #             pathways]
        nested_list = [processed_disease,
                processed_gene,
                processed_celltype,
                pathways]
        source_map = {
            0: "disease",
            1: "gene",
            2: "cell type",
            3: "pathways"
        }
        flattened_list_with_source = [(item, source_map[i]) for i, sublist in enumerate(nested_list) for item in sublist]
        if len(flattened_list_with_source) == 0:
            continue
        response = get_response_finalize([item for item, _ in flattened_list_with_source])
        response = response.lower()
        # if "\n" in response or response.startswith("- "):
        #     filtered_entities_with_source = [(item, source) for item, source in flattened_list_with_source if item in [item.strip('- ') for item in response.split('\n')]]
        # else:
        #     filtered_entities_with_source = [(item, source) for item, source in flattened_list_with_source if item in response]
        # terms = re.findall(r'\* ([\w()]+)', response)
        # filtered_entities_with_source = [(item, source) for item, source in flattened_list_with_source if item in terms]
        # terms = [line.strip("• ").strip() for line in response.splitlines() if line.strip().startswith("•")]
        # filtered_entities_with_source = []
        # for item, source in flattened_list_with_source:
        #     match, score, _ = process.extractOne(item, terms, scorer=fuzz.partial_ratio)
        #     if score >= 85:
        #         filtered_entities_with_source.append((item, source))
        # match = re.search(r"\[.*\]", response)
        match = re.search(r"\[.*\]", response, re.DOTALL)
        if match:
            try:
                list_str = match.group(0)  # Get the matched list string
                parsed_list = ast.literal_eval(list_str)  # Safely evaluate to list of tuples
                filtered_entities_with_source = dict(parsed_list)
            except (SyntaxError, ValueError) as e:
                match = re.search(r"\[\((.*?)\)\]", response)
                inner = match.group(1) 
                parts = [f"'{p.strip()}'" for p in inner.split(',')]
                safe_list_str = f"[({', '.join(parts)})]"
                parsed_list = ast.literal_eval(safe_list_str)
                filtered_entities_with_source = dict(parsed_list)
        else:
            tuple_list = ast.literal_eval(response)
            filtered_entities_with_source = dict(tuple_list)

        output = "/Volumes/she4/knowledgegraph/result/new_extracted_entities_from_NK_abstracts/"
        with open(os.path.join(output, pmid), 'w') as file:
            for item, source in filtered_entities_with_source.items():
                file.write(f"{item.replace('- ', '')}\t{source}\n")

        # output = "/Volumes/she4/knowledgegraph/result/extracted_entities_from_NK_abstracts/"
        # with open(os.path.join(output,pmid), 'w') as file:
        #     for item in filtered_entities:
        #         # file.write(f"{item}\n")
        #         file.write(f"{item.replace('- ', '')}\n")

        abstract = text
        # entities = [item[0] for item in filtered_entities_with_source]
        entities = [item for item in filtered_entities_with_source.keys()]
        relationships = get_relationships(abstract, entities)
        if len(relationships) != 0:
            prefix1 = "Here is the list of output tuples:\n\n("
            prefix2 = "Here is my response:\n\n("
            prefix3 = "here is the list of informative terms:\n\n("
            prefix4 = "Here is the list of valid biomedical pathways:\n\n("
            first_tuple = relationships[0]
            if first_tuple[0].startswith(prefix1) or first_tuple[0].startswith(prefix2) or first_tuple[0].startswith(prefix3) or first_tuple[0].startswith(prefix4):
                if first_tuple[0].startswith(prefix1):
                        matched_prefix = prefix1
                elif first_tuple[0].startswith(prefix2):
                    matched_prefix = prefix2
                elif first_tuple[0].startswith(prefix3):
                    matched_prefix = prefix3
                elif first_tuple[0].startswith(prefix4):
                    matched_prefix = prefix4
                cleaned_first = (first_tuple[0].replace(matched_prefix, '').strip(), first_tuple[1], first_tuple[2])
                cleaned_list = [cleaned_first] + relationships[1:]
            else:
                cleaned_list = relationships
            relationships = cleaned_list

        df = pd.DataFrame(relationships, columns=['Outgoing', 'Incoming', 'Relationship'])
        df['abstract_id'] = pmid

        entity_source_dict = dict(filtered_entities_with_source)
        def get_source(entity):
            return entity_source_dict.get(entity, 'unknown')
        df['outgoing_source'] = df['Outgoing'].apply(get_source)
        df['incoming_source'] = df['Incoming'].apply(get_source)

        # normalized_df = normalize_dataframe(df, model)
        output = "/Volumes/she4/knowledgegraph/result/new_extracted_entities_from_NK_abstracts/"
        df.to_csv(os.path.join(output,pmid+'_relationship.csv'), index=False)
        print(number)
        print("processed: "+str(processed))

### Normalize the gene names for each csv file - 2.normalization.ipynb

In [ ]:
import pandas as pd
import glob
import os
from functools import lru_cache

relationship_client = Groq(api_key='')

def generate_prompt_gene_normalize(gene_entity):
    system_prompt = """
    You are a biologist who are very familiar with gene names and their official hgnc symbol.
    """
    user_prompt = f"""
    I will provide you with an entity that is likely a gene.
    Entity: {gene_entity}

    For this gene, output its official hgnc symbol only, with no additional texts!

    Important rules to follow:
    - If this entity does not look like a gene, then output "doesntlooklikegene" !! don't add new texts.
    - Some provided entities are the full name of a gene without space, so be careful.

    **Remember:**
    Output only the official hgnc symbol, don't add new texts!!
    Examples: (gene_entity=fcgammareceptor3a)
    - FCGR3A
    Examples: (gene_entity=tumorprogression)
    - doesntlooklikegene
    """
    return system_prompt, user_prompt

@lru_cache(maxsize=None)
def get_gene_synonyms(gene_entity):
    system_prompt, user_prompt = generate_prompt_gene_normalize(gene_entity)
    response = relationship_client.chat.completions.create(
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            }
        ],
        model="llama3-70b-8192",
        max_tokens=8000,
        temperature=0
    )
    res_strip = response.choices[0].message.content
    lines = res_strip.strip().split('\n')
    synonyms_list = [line.lstrip('- ').strip() for line in lines]
    return synonyms_list[0].upper()

def remove_single_character_entries(df):
    """Remove rows where 'Outgoing' or 'Incoming' has only one character."""
    return df[(df['Outgoing'].str.len() > 1) & (df['Incoming'].str.len() > 1)]

def normalize_gene_dataframe(df, unique_outgoing, unique_incoming):
    def conditional_normalize(value, column):
        if df.loc[df[column] == value, f'{column.lower()}_source'].iloc[0] in ['gene', 'unknown']:
            return get_gene_synonyms(value)
        else:
            return value

    normalized_outgoing = {value: conditional_normalize(value, 'Outgoing') for value in unique_outgoing}
    normalized_incoming = {value: conditional_normalize(value, 'Incoming') for value in unique_incoming}

    df['Normalized_Outgoing'] = df['Outgoing'].map(normalized_outgoing)
    df['Normalized_Incoming'] = df['Incoming'].map(normalized_incoming)

    return df


In [ ]:
noname = 0
folder_path = "/Volumes/she4/knowledgegraph/result/new_extracted_entities_from_NK_abstracts"
csv_files = glob.glob(os.path.join(folder_path, "*relationship.csv"))
start_index = 0
for i, file_path in enumerate(csv_files[start_index:], start_index + 1):
    print(f"Processing file: {file_path}")
    df = pd.read_csv(file_path)
    if "Outgoing" not in df.columns:
        noname += 1 
        continue
    df = remove_single_character_entries(df)

    # Get unique values
    unique_outgoing = df['Outgoing'].unique()
    unique_incoming = df['Incoming'].unique()

    # Normalize the dataframe
    normalized_df = df.drop_duplicates(subset=['Outgoing', 'Incoming'], keep='first')
    normalized_df = normalize_gene_dataframe(df, unique_outgoing, unique_incoming)

    normalized_df.to_csv(file_path, index=False)
    print(f"Normalized data saved to: {file_path}")
    print(i)

print("All files processed.")

# Assemble the KG

### Build the graph

In [ ]:
import os
import pandas as pd
import csv
import re
import networkx as nx
import numpy as np
from collections import defaultdict
from gensim.models import Word2Vec
from gensim.models.keyedvectors import KeyedVectors
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap, Normalize
from adjustText import adjust_text

def is_valid_file(filename):
    """Check if the file is a valid CSV and not hidden."""
    return not filename.startswith('.') and filename.endswith('.csv')

def build_weighted_network(folder_path):
    G = nx.DiGraph()
    edge_counts = defaultdict(int)
    edge_file_counts = defaultdict(set)
    node_types = {}
    edge_sources = defaultdict(set)  # New dictionary to store sources for each edge
    
    for filename in os.listdir(folder_path):
        if is_valid_file(filename):
            d =  pd.read_csv(os.path.join(folder_path, filename)) # this is needed for macrophage
            if 'Unnamed: 0' not in d.columns: # this is needed for macrophage
                if "Normalized_Outgoing" in d.columns: # this is needed for macrophage
                    with open(os.path.join(folder_path, filename), 'r') as csvfile:
                        reader = csv.DictReader(csvfile)
                        for row in reader:
                            # outgoing = row['Normalized_Outgoing']
                            # incoming = row['Normalized_Incoming']
                            if ((row['Normalized_Outgoing'] == "DOESNTLOOKLIKEGENE" and row['outgoing_source'] == "gene") or
                            (row['Normalized_Incoming'] == "DOESNTLOOKLIKEGENE" and row['incoming_source'] == "gene")):
                                continue  
                            outgoing = row['Normalized_Outgoing'] if row['outgoing_source'] == "gene" else row['Outgoing']
                            incoming = row['Normalized_Incoming'] if row['incoming_source'] == "gene" else row['Incoming']

                            relationship = row['Relationship'].lower()
                            abstract_id = row['abstract_id']  # Get the abstract_id
                            
                            # Store node types
                            node_types[outgoing] = row['outgoing_source']
                            node_types[incoming] = row['incoming_source']
                            
                            G.add_node(outgoing, node_type=row['outgoing_source'])
                            G.add_node(incoming, node_type=row['incoming_source'])
                            edge_counts[(outgoing, incoming, relationship)] += 1
                            edge_file_counts[(outgoing, incoming, relationship)].add(filename)
                            
                            # Store the abstract_id as a source for this edge
                            edge_sources[(outgoing, incoming, relationship)].add(abstract_id)
            
    for (outgoing, incoming, relationship), count in edge_counts.items():
        G.add_edge(outgoing, incoming, 
                   # weight=count, 
                   weight=len(edge_file_counts[(outgoing, incoming, relationship)]),
                   relationship=relationship, 
                   file_count=len(edge_file_counts[(outgoing, incoming, relationship)]),
                   sources=list(edge_sources[(outgoing, incoming, relationship)]))  # Add sources to edge attributes
    
    return G

def visualize_network(G):
    """Visualize the network with improved layout and less overlap."""
    # Use Fruchterman-Reingold layout with adjusted parameters
    pos = nx.spring_layout(G, k=0.5, iterations=50, seed=820)
    
    plt.figure(figsize=(20, 16))
    
    # Create color maps for activate and inhibit
    activate_cmap = LinearSegmentedColormap.from_list("", ["#E6FFE6", "#00FF00"])
    inhibit_cmap = LinearSegmentedColormap.from_list("", ["#FFE6E6", "#FF0000"])
    
    # Separate activate and inhibit edges
    activate_edges = [(u, v) for (u, v, d) in G.edges(data=True) if d['relationship'] == 'activate']
    inhibit_edges = [(u, v) for (u, v, d) in G.edges(data=True) if d['relationship'] == 'inhibit']
    
    # Get edge weights
    activate_weights = [G[u][v]['file_count'] for (u, v) in activate_edges]
    inhibit_weights = [G[u][v]['file_count'] for (u, v) in inhibit_edges]
    
    # Normalize weights
    all_weights = activate_weights + inhibit_weights
    weight_max = max(all_weights) if all_weights else 1
    weight_min = min(all_weights) if all_weights else 0
    normalize = lambda x: (x - weight_min) / (weight_max - weight_min) if weight_max > weight_min else 0.5

    # Draw activate edges
    nx.draw_networkx_edges(G, pos, edgelist=activate_edges, edge_color=[normalize(w) for w in activate_weights],
                           edge_cmap=activate_cmap, width=2, alpha=0.7)
    
    # Draw inhibit edges
    nx.draw_networkx_edges(G, pos, edgelist=inhibit_edges, edge_color=[normalize(w) for w in inhibit_weights],
                           edge_cmap=inhibit_cmap, width=2, alpha=0.7)
    
    # Calculate node sizes based on degree
    degrees = dict(G.degree())
    node_sizes = [300 * (1 + degrees[node] / max(degrees.values())) for node in G.nodes()]
    
    # Draw nodes
    nx.draw_networkx_nodes(G, pos, node_color='lightblue', node_size=node_sizes, alpha=0.8)
    
    # Add labels with custom positioning
    labels = {node: node for node in G.nodes()}
    texts = []
    for node, (x, y) in pos.items():
        texts.append(plt.text(x, y, node, fontsize=12, ha='center', va='center'))
    
    # Adjust label positions to avoid overlaps
    adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))
    
    # Create legend
    activate_line = Line2D([0], [0], color="#00FF00", lw=2, label='Activate')
    inhibit_line = Line2D([0], [0], color="#FF0000", lw=2, label='Inhibit')
    plt.legend(handles=[activate_line, inhibit_line], 
               loc='upper left', bbox_to_anchor=(1, 1), title="Relationship Types")
    
    # Add colorbar
    sm = plt.cm.ScalarMappable(cmap=LinearSegmentedColormap.from_list("", ["#FFFFFF", "#00FF00"]),
                               norm=plt.Normalize(vmin=weight_min, vmax=weight_max))
    sm.set_array([])
    cbar = plt.colorbar(sm, label='Number of Files', orientation='horizontal', pad=0.1, aspect=30, shrink=0.5)
    
    plt.title("Normalized Weighted Network (Word2Vec)")
    plt.axis('off')
    plt.tight_layout()
    plt.show()

# # Main execution
folder_path = '/Volumes/she4/knowledgegraph/result/extracted_entities_from_NK_abstracts'
weighted_network = build_weighted_network(folder_path)